<a href="https://colab.research.google.com/github/aounraza379/AutiSense-AI/blob/main/autisense_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install necessary libraries
!pip install -q groq gradio python-dotenv openai-whisper gTTS
# 2. Install ffmpeg (Required for Whisper audio processing)
!apt install ffmpeg -y -q

In [ ]:
import os
import whisper
import gradio as gr
from gtts import gTTS
from groq import Groq
from dotenv import load_dotenv
from google.colab import userdata

# 1. SAFE API KEY LOADING (COLAB WAY)
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in Colab Secrets")

client = Groq(api_key=GROQ_API_KEY)

# Load Whisper once to prevent repeated memory overhead
# 'tiny' is used for HF Spaces CPU efficiency; 'base' for better accuracy
try:
    stt_model = whisper.load_model("tiny")
except Exception as e:
    print(f"STT Initialization Error: {e}")

# 2. STABLE MEMORY SYSTEM (With Reset & Caps)
class SessionState:
    def __init__(self):
        self.reset()

    def reset(self):
        self.memory = {"low_politeness": 0, "low_clarity": 0, "no_questions": 0}
        self.history = []
        return [], "Memory Reset! Ready for a new session.", None

    def update(self, text):
        t = text.lower().strip()
        words = t.split()
        # Cap counters at 5 to prevent "noise accumulation"
        if (len(t) <= 2 or len(words) < 2) and self.memory["low_clarity"] < 5:
            self.memory["low_clarity"] += 1
        if ("please" not in t and "thank" not in t) and self.memory["low_politeness"] < 5:
            self.memory["low_politeness"] += 1
        if "?" not in t and self.no_questions_check(t) and self.memory["no_questions"] < 5:
            self.memory["no_questions"] += 1

    def no_questions_check(self, text):
        question_words = ["how", "what", "where", "why", "can"]
        return not any(word in text for word in question_words)

    def get_profile(self):
        tags = []
        if self.memory["low_politeness"] > 1: tags.append("- Forgets politeness.")
        if self.memory["low_clarity"] > 1: tags.append("- Very short answers.")
        if self.memory["no_questions"] > 1: tags.append("- Rarely asks questions.")
        return "\n".join(tags) if tags else "Communicating well."

state_manager = SessionState()

# 3. ROBUST UTILITIES (Error Handling Layer)
def safe_transcribe(audio_path):
    if not audio_path: return ""
    try:
        return stt_model.transcribe(audio_path)["text"]
    except: return "[Unintelligible audio]"

def safe_tts(text):
    try:
        tts = gTTS(text=text, lang='en')
        path = "response.mp3"
        tts.save(path)
        return path
    except: return None

# 4. CORE HANDLER (Optimized & Non-Blocking)
def process_all(audio_path, history):
    user_text = safe_transcribe(audio_path)

    if not user_text.strip() or user_text == "[Unintelligible audio]":
        return history, "I didn't hear that. Could you try again? 😊", None

    # Logic updates
    state_manager.update(user_text)
    feedback = "Keep going! You're doing great! ✨" # Default

    # Priority Feedback
    if "please" not in user_text.lower():
        feedback = "Try saying 'please' to be extra polite! 🙏"
    elif len(user_text.split()) < 3:
        feedback = "Great! Try using a slightly longer sentence. 😊"

    # AI Request with Error Handling
    profile = state_manager.get_profile()
    prompt = f"Friendly shopkeeper. Simple English. 1-2 sentences. Profile: {profile}"

    messages = [{"role": "system", "content": prompt}]
    for m in history:
        messages.append({"role": m["role"], "content": m["content"]})
    messages.append({"role": "user", "content": user_text})

    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            timeout=10
        )
        ai_text = chat_completion.choices[0].message.content
    except:
        ai_text = "I'm sorry, my shop is a bit busy! Can you say that again?"

    audio_res = safe_tts(ai_text)
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": ai_text})

    return history, feedback, audio_res

# 5. REDESIGNED UI (Production Layout)
with gr.Blocks(theme=gr.themes.Soft(), title="AutiSense AI") as demo:
    gr.Markdown("## AutiSense AI: Conversation Trainer")
    gr.Markdown("Practice social skills in a safe, friendly environment.")

    with gr.Row():
        # LEFT PANEL: Primary Interaction
        with gr.Column(scale=2):
            chat_box = gr.Chatbot(label="Shop Counter", type="messages", height=450)
            with gr.Group():
                audio_in = gr.Audio(label="Your Voice (Click Mic)", type="filepath", sources=["microphone"])
                audio_out = gr.Audio(label="Shopkeeper's Voice", autoplay=True, visible=False)

            with gr.Row():
                reset_btn = gr.Button("Reset Session", variant="secondary")
                stop_btn = gr.Button("Stop Recording", variant="stop")

        # RIGHT PANEL: Coaching & Progress
        with gr.Column(scale=1):
            feedback_display = gr.Label(label="Coaching Feedback", value="Welcome! Say 'Hello' to start.")
            with gr.Accordion("Session Insights", open=True):
                gr.Markdown("The AI adapts its personality based on your habits.")
                memory_view = gr.Textbox(label="Current Profile Status", value="Waiting for conversation...", interactive=False)

    # EVENT BINDING
    audio_in.stop_recording(process_all, [audio_in, chat_box], [chat_box, feedback_display, audio_out])

    # Update memory view separately to keep UI clean
    audio_in.stop_recording(lambda: state_manager.get_profile(), None, memory_view)

    reset_btn.click(state_manager.reset, None, [chat_box, feedback_display, audio_out])

if __name__ == "__main__":
    demo.launch()